In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
#retstart runtime after

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu!!")

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q sentence-transformers

### Upload Files

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving processed_documents_recursive.zip to processed_documents_recursive.zip


In [ ]:
import zipfile
with zipfile.ZipFile("processed_documents_recursive.zip") as z:
    z.extractall("data")
# adjust if your zip nests differently — check with the next line
!find data -name "metadata.json"

data/processed_documents_recursive/metadata.json


In [ ]:
!rm -rf data/processed_documents_fixed/

### loader

In [ ]:
import json
from pathlib import Path

PROCESSED = Path("data/processed_documents_recursive")   # match what `find` showed

def load_corpus():
    master = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    corpus, skipped = [], 0
    for fname, meta in master.items():
        if meta.get("is_duplicate"):
            skipped += 1
            continue
        stem = Path(fname).stem
        dj = PROCESSED / f"{stem}.json"
        if not dj.exists():
            continue
        data = json.loads(dj.read_text(encoding="utf-8"))
        for c in data["chunks"]["chunks"]:
            t = c["text"].strip()
            if not t:
                continue
            corpus.append({
                "chunk_id": f"{stem}__{c['chunk_index']}",
                "text": t, "source_doc": fname,
                "language": meta.get("primary_language"),
                "token_count": c.get("token_count"),
            })
    print(f"[loader] {len(corpus)} chunks | {skipped} dupe docs skipped")
    return corpus

corpus = load_corpus()

[loader] 7325 chunks | 10 dupe docs skipped


In [ ]:
!pip uninstall -y sentence-transformers huggingface-hub InstructorEmbedding transformers
!pip install sentence-transformers==2.2.2 huggingface_hub==0.20.3 InstructorEmbedding

### embed function

In [ ]:
import numpy as np, json, time
from sentence_transformers import SentenceTransformer
from google.colab import files

def embed_and_download(model_name, hf_id, prefix="", batch_size=64):
    print(f"\n=== {model_name} ===")
    model = SentenceTransformer(hf_id, device="cuda")

    texts = [prefix + c["text"] for c in corpus]   # prefix applied here
    ids   = [c["chunk_id"] for c in corpus]

    t = time.time()
    vecs = model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True)
    dt = time.time() - t
    vecs = vecs.astype(np.float32)

    np.save(f"{model_name}.npy", vecs)
    with open(f"{model_name}.meta.json", "w") as f:
        json.dump({"model": model_name, "hf_id": hf_id, "prefix": prefix,
                   "dim": int(vecs.shape[1]), "count": int(vecs.shape[0]),
                   "chunk_ids": ids, "encode_seconds": round(dt, 1)}, f)

    print(f"done: {vecs.shape} in {dt:.1f}s ({1000*dt/len(texts):.0f} ms/chunk)")
    files.download(f"{model_name}.npy")
    files.download(f"{model_name}.meta.json")

## BGE
english baseline

In [ ]:
embed_and_download("bge-large", "BAAI/bge-large-en-v1.5", prefix="")

## BGE-M3
multilingual, no prefix, big context window

most versatile and advanced model in this list !

In [ ]:
embed_and_download("bge-m3", "BAAI/bge-m3", prefix="")

## E5
learn to pull similar texts together and push dissimilar ones apart in the vector space, requires prefix

In [ ]:
embed_and_download("e5-large", "intfloat/e5-large-v2", prefix="passage: ")

## Multilingual E5

In [ ]:
embed_and_download("multilingual-e5-large", "intfloat/multilingual-e5-large", prefix="passage: ")

# Embedding Query File

In [4]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from google.colab import files
import time

# 1. Upload your eval_queries.json file
uploaded = files.upload()

Saving eval_queries.json to eval_queries.json


In [5]:

# 2. Load the queries
queries = json.loads(open("eval_queries.json", encoding="utf-8").read())["queries"]
texts = [q["query"] for q in queries]

print(f"Loaded {len(queries)} queries")

# 3. Define models: (name, hf_id, query_prefix)
MODELS = [
    ("bge-large", "BAAI/bge-large-en-v1.5", ""),
    ("bge-m3", "BAAI/bge-m3", ""),
    ("e5-large", "intfloat/e5-large-v2", "query: "),
    ("multilingual-e5-large", "intfloat/multilingual-e5-large", "query: "),
]

# 4. Embed with each model
for name, hf_id, prefix in MODELS:
    print(f"\n=== {name} (prefix={prefix!r}) ===")
    model = SentenceTransformer(hf_id, device="cuda")
    t = time.time()

    # Add prefix to each query
    prefixed_texts = [prefix + x for x in texts]

    # Encode
    vecs = model.encode(
        prefixed_texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    print(f"  Shape: {vecs.shape}")
    print(f"  Time: {time.time()-t:.0f}s")

    # Save and download
    np.save(f"{name}.npy", vecs)
    files.download(f"{name}.npy")

Loaded 45 queries

=== bge-large (prefix='') ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  Shape: (45, 1024)
  Time: 1s


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...


=== multilingual-e5-large (prefix='query: ') ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  Shape: (45, 1024)
  Time: 0s


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>